# Avance 6 — Conclusiones Clave

## Proyecto Integrador – Equipo 22

Integrantes:
* María Virginia Mendizábal Miranda - A01796588
* Gianmel Joannelly Hernández Tosta - A01795919
* Sofía Ordaz López - A01173717

**Proyecto:** Análisis de biomarcadores lipidómicos y metabolómicos mediante modelos de *machine learning* interpretativo para clasificación multiclase de cáncer colorrectal.

**Fecha de entrega:** 14 de junio de 2026

---

## 1. Introducción y Contexto

Este avance cierra el ciclo CRISP-ML(Q) del proyecto integrador, cuyo objetivo fue construir un clasificador multiclase supervisado capaz de distinguir entre tres grupos diagnósticos —**CTRL** (controles sanos), **AA** (adenomas avanzados) y **CRC** (cáncer colorrectal confirmado)— utilizando biomarcadores lipidómicos y metabolómicos como *features*, y cuantificar la contribución predictiva de estos biomarcadores respecto a las variables clínicas estándar.

**Objetivo clínico central:** Detectar adenomas avanzados (AA) como ventana de intervención preventiva, permitiendo su remoción antes de la progresión a CRC, mientras se minimizan colonoscopías innecesarias.

A lo largo de cinco avances previos se recorrieron las fases del marco CRISP-ML(Q):

| Avance | Fase CRISP-ML(Q) | Entregable principal |
|---|---|---|
| 1 | Comprensión del negocio y datos | EDA completo, diagnóstico de calidad, ranking supervisado de variables |
| 2 | Preparación de datos | Imputación validada, transformación Box-Cox, selección de características (4 matrices candidatas) |
| 3 | Modelado (baseline) | Regresión logística multinomial como línea base (F1 macro = 0.628) |
| 4 | Modelado y evaluación | Comparación de 6 algoritmos individuales; selección de Decision Tree (F1 macro held-out = 0.681) |
| 5 | Modelado avanzado (ensembles) | 8 modelos ensemble; selección de Bagging Tree sobre X_intersect (F1 macro held-out = 0.855) y análisis complementario X_full Top-K (Top 15 conserva Recall AA = 0.833) |

En este **Avance 6** se presentan las **conclusiones clave** del proyecto, organizadas en los siguientes ejes:

1. Evaluación final del modelo seleccionado contra los criterios de éxito definidos en Fase 0
2. Decisión de implementación: viabilidad para piloto controlado
3. Hallazgos del negocio y recomendaciones accionables por *stakeholder*
4. Comparativa de proveedores cloud y arquitectura para piloto en entorno productivo experimental
5. Reflexión sobre el proceso CRISP-ML(Q) y lecciones aprendidas

---

## 2. Evaluación Final del Modelo Seleccionado vs. Criterios de Éxito (Fase 0)

### 2.1 Resumen del modelo seleccionado

El modelo final seleccionado es **Bagging Tree** (ensemble homogéneo), entrenado sobre la matriz **X_intersect** (16 *features*: 12 lípidos + 2 variables clínicas + 2 categóricas).

**Configuración óptima (GridSearchCV, 5-fold estratificado sobre X_dev):**

```python
BaggingClassifier(
    estimator=DecisionTreeClassifier(class_weight='balanced', max_depth=None),
    n_estimators=100,
    max_samples=0.8,
    max_features=0.6,
    random_state=42
)
```

**Justificación de la selección sobre otros candidatos:**
- Superior F1 macro held-out (0.855) frente a Random Forest (0.810), Gradient Boosting (0.809) y modelos heterogéneos (Stacking 0.767, Voting 0.721).
- Mejor detección de AA (F1 = 0.78), la clase clínicamente prioritaria.
- Coherencia con la familia de árboles de decisión seleccionada en Avance 4, preservando interpretabilidad.
- AUC macro de 0.945, indicando excelente poder discriminatorio.
- Análisis complementario Top-K: las 15 variables más importantes de X_full conservaron el mismo Recall AA (0.833) y F1 AA (0.833) que X_full completo, reduciendo la dimensionalidad de 131 a 15 predictores.

### 2.2 Síntesis: Criterios de Éxito Fase 0 vs. Resultados Finales

La siguiente tabla resume de forma directa el cumplimiento de las metas definidas desde la Fase 0 del proyecto:

| Criterio de éxito Fase 0 | Resultado final | Estado |
|---|---|---|
| Superar baseline F1 macro (0.628) | F1 macro = **0.855** (+36.1%) | **Cumplido** |
| Mejorar detección de adenomas avanzados (AA) | F1 AA = **0.78** (vs. 0 en PLS-DA de referencia) | **Cumplido** |
| Mantener interpretabilidad del modelo | Bagging Tree (árboles interpretables) + SHAP propuesto | **Cumplido** |
| Reducir dimensionalidad sin perder desempeño | 16 variables en el modelo final X_intersect; además, Top 15 derivado de X_full conserva Recall AA = 0.833 con 15 predictores | **Cumplido** |
| Mantener utilidad biomédica | Biomarcadores lipidómicos identificados con plausibilidad biológica (CE(20:5), TG(51:4)) | **Cumplido** |

### 2.3 Tabla comparativa detallada: Modelo final vs. Criterios de Éxito

Los criterios de éxito se definieron progresivamente a lo largo de las fases del proyecto. A continuación se contrastan con los resultados finales del Bagging Tree:

| # | Criterio de Éxito (Fase 0 / Objetivos) | Umbral o Referencia | Resultado Modelo Final | Brecha | Cumplimiento |
|---|---|---|---|---|---|
| **CE1** | F1 macro superior al baseline aleatorio (1/3) | ≥ 0.333 | **0.855** | **+0.522** | Superado ampliamente |
| **CE2** | F1 macro superior al baseline de Regresión Logística (Avance 3) | ≥ 0.628 | **0.855** | **+0.227 (+36.1%)** | Superado |
| **CE3** | Mejora sobre modelo individual Decision Tree (Avance 4) | ≥ 0.681 | **0.855** | **+0.174 (+25.5%)** | Superado |
| **CE4** | No colapso de ninguna clase a F1 ≈ 0 (referencia PLS-DA del paper: F1 AA = 0) | F1 AA > 0 | **F1 AA = 0.78** | **+0.78** | Superado |
| **CE5** | Detección de AA (recall) superior al 50% (umbral mínimo clínicamente relevante) | Recall AA ≥ 0.50 | **Recall AA = 0.75** | **+0.25 (+50%)** | Superado |
| **CE6** | Reducción de varianza respecto al árbol único (Objetivo 3.5) | Δ F1 > 0 | **+0.174** | — | Superado |
| **CE7** | Precisión en CTRL alta para minimizar colonoscopías innecesarias | Precisión CTRL alta | **Precisión CTRL = 1.00** | — | Cumplido |
| **CE8** | AUC ROC macro > 0.80 (discriminación clínicamente útil) | ≥ 0.80 | **AUC = 0.945** | **+0.145** | Superado |
| **CE9** | Parsimonia: modelo interpretable con número reducido de *features* | < 30 features | **16 features** | — | Cumplido |
| **CE10** | Reproducibilidad CRISP-ML(Q): pipeline trazable y replicable | Documentación completa | Pipeline completo (Avances 1–6) | — | Cumplido |
| **CE11** | Preservar el beneficio clínico observado con X_full usando menos predictores | Recall AA comparable a X_full con <30 features | **Top 15: Recall AA = 0.833; F1 AA = 0.833** | 116 variables menos que X_full | Cumplido como hallazgo exploratorio |

### 2.4 Análisis de brechas

#### Brechas positivas (criterios superados con margen)

1. **F1 macro (+36.1% sobre baseline):** La mejora de 0.628 a 0.855 valida la hipótesis central del proyecto: los ensembles de árboles explotan eficazmente la señal lipidómica dispersa en múltiples variables, reduciendo la varianza inherente a un árbol único. El salto de +0.174 respecto al Decision Tree individual (Avance 4) confirma que la agregación de 100 árboles con *bootstrap* estabiliza las predicciones.

2. **Detección de AA (F1 = 0.78 vs. histórico de 0):** Este es el resultado más significativo clínicamente. El modelo PLS-DA de referencia (Albóniga et al., 2025) clasifica **cero** muestras de AA correctamente. Nuestro modelo detecta el 75% de los adenomas avanzados con un F1 de 0.78, abriendo una ventana real de prevención.

3. **Precisión CTRL = 1.00 en held-out:** La clase CTRL alcanzó precisión de 1.00 en el conjunto held-out (n=16 controles), lo que sugiere una baja tendencia del modelo a generar falsos positivos en controles sanos dentro de esta partición de prueba. En contexto de screening, esto es favorable para minimizar colonoscopías innecesarias, aunque debe confirmarse con tamaño muestral mayor.

4. **AUC macro = 0.945:** El poder discriminatorio del modelo es excelente en las tres clases (AA: 0.902, CRC: 0.949, CTRL: 0.984), lo que indica robustez incluso con diferentes umbrales de decisión.

5. **Concentración de señal en Top 15 derivado de X_full:** El análisis posterior a la retroalimentación mostró que el beneficio observado en X_full para AA no requiere las 131 variables originales. El subconjunto Top 15 mantuvo el mismo Recall AA (0.833) y F1 AA (0.833) que X_full completo, con F1 macro prácticamente idéntico (0.839).

#### Brechas negativas y cautelas

1. **Discrepancia CV → held-out (-0.053):** El F1 held-out (0.855) supera al F1 de validación cruzada (0.802). Esto es atípico y se atribuye a la varianza inherente de un set de prueba pequeño (n=43). **La estimación más conservadora y fiable del rendimiento externo es el F1 CV de 0.802**, no el 0.855. Un intervalo realista sería F1 ∈ [0.78, 0.86] considerando la variabilidad muestral.

2. **Recall AA = 0.75 en el modelo final X_intersect (25% de AA no detectados):** Tres de cada doce pacientes con adenoma avanzado no fueron identificados. En un contexto clínico donde la detección temprana salva vidas, esta tasa de falsos negativos para AA requiere atención. El análisis Top-K mostró una alternativa exploratoria: Top 15 derivado de X_full aumenta Recall AA a 0.833, aunque con menor F1 macro global que X_intersect.

3. **Tamaño muestral limitado (n=211, test=43):** Las métricas held-out fluctúan ±0.02–0.05 puntos F1 dependiendo de la composición específica del split. Validación externa con cohorte independiente es necesaria antes de cualquier aplicación clínica.

4. **Overfitting en entrenamiento (Train F1 = 1.00):** El modelo ajusta perfectamente los datos de entrenamiento, lo que indica que los árboles sin poda (`max_depth=None`) memorizan el set de desarrollo. Aunque la regularización vía *bagging* (80% muestras, 60% features por árbol) mitiga parcialmente este efecto, una brecha train-CV de ~0.20 sugiere margen para mejorar la regularización.

5. **Análisis Top-K exploratorio:** Los subconjuntos Top 10/15/20/30 se evaluaron sobre held-out para responder la observación docente, pero no fueron sometidos a una nueva validación cruzada completa. Por ello, el hallazgo Top 15 debe interpretarse como evidencia preliminar para iteraciones futuras, no como reemplazo definitivo del modelo seleccionado.

### 2.5 Evolución del rendimiento a lo largo del proyecto

La siguiente tabla traza la evolución del F1 macro a lo largo de los avances, demostrando mejora consistente:

| Avance | Modelo | F1 Macro (CV) | F1 Macro (Held-out) | F1 AA | Recall AA | Mejora acumulada |
|---|---|---|---|---|---|---|
| <nobr>3 (Baseline)</nobr> | Regresión Logística | 0.628 | — | 0.38 | 0.47 | Referencia |
| <nobr>4 (Individual)</nobr> | Decision Tree | 0.707 | 0.681 | ~0.55 | 0.50 | +12.6% CV |
| <nobr>5 (Ensemble)</nobr> | **Bagging Tree** | **0.802** | **0.855** | **0.78** | **0.75** | **+27.7% CV** |

**Observaciones clave de la evolución:**

- La mejora más grande ocurrió en el paso de modelo individual a ensemble (+0.095 en CV, +0.174 en held-out), validando el Objetivo 3.5 del proyecto.
- La clase AA experimentó la mayor ganancia relativa: de F1 = 0.38 (baseline) a F1 = 0.78 (final), un incremento de **+105%**.
- El recall de CRC alcanzó 0.93 en el modelo final, indicando que el 93% de los pacientes con cáncer confirmado son detectados correctamente.

#### Análisis complementario Top-K posterior a la retroalimentación

| Configuración | n_features | F1 Macro (Held-out) | F1 AA | Recall AA | Lectura |
|---|---:|---:|---:|---:|---|
| **X_intersect + Bagging Tree** | 16 | **0.855** | 0.783 | 0.750 | Mejor desempeño global; modelo final |
| **X_full + Bagging Tree** | 131 | 0.839 | 0.833 | **0.833** | Mejora detección de AA con alta dimensionalidad |
| **X_full Top 15 + Bagging Tree** | 15 | 0.839 | **0.833** | **0.833** | Conserva el beneficio de AA con 89% menos variables |

Este análisis responde a la observación de la Dra. Grettel: el incremento en AA observado con X_full no depende de las 131 variables completas, sino que puede preservarse con un subconjunto reducido de 15 predictores.


---

## 3. Decisión de Implementación del Modelo

### 3.1 Veredicto de implementación

> **No se recomienda implementación clínica inmediata como herramienta diagnóstica autónoma.** Sí se recomienda una **implementación piloto controlada** como sistema de apoyo a la investigación y *triage* experimental, sujeto a validación externa, revisión ética y validación clínica prospectiva.

**El sistema debe considerarse exclusivamente como herramienta de apoyo experimental y no como dispositivo médico validado para toma de decisiones clínicas.**

Esta decisión refleja madurez metodológica: el modelo demuestra rendimiento prometedor, pero aún no ha superado las barreras necesarias para uso clínico definitivo.

### 3.2 Justificación detallada

| Pregunta clave | Respuesta | Justificación |
|---|---|---|
| ¿El rendimiento es suficiente para producción clínica definitiva? | **No aún.** Sí para piloto investigativo controlado. | F1 macro = 0.855 y AUC = 0.945 son métricas prometedoras, pero se validaron sobre n=43 muestras de una sola cohorte (Ourense). Para uso clínico definitivo se recomienda una validación externa multicéntrica, idealmente con n ≥ 200. |
| ¿Existe margen de mejora técnica? | **Sí.** | Train F1 = 1.00 indica memorización; la regularización (`max_depth` limitado, `min_samples_leaf`), técnicas de explicabilidad (SHAP), calibración de umbrales por clase, *oversampling* de AA y validación formal del subconjunto Top 15 derivado de X_full son mejoras concretas implementables. |
| ¿Qué se recomienda implementar ahora? | **Prototipo de investigación.** | Pipeline reproducible, API segura con datos anonimizados, monitoreo de *drift*, trazabilidad CRISP-ML(Q), explicación por paciente y control de versiones del modelo. |
| ¿Qué se debe cumplir antes de uso clínico real? | **Cuatro requisitos previos.** | (1) Validación externa multicéntrica, (2) autorización del comité de ética, (3) evaluación costo-beneficio formal, (4) revisión y aprobación por equipo médico especialista. |

### 3.3 Condiciones para transición a producción clínica

El modelo podría avanzar hacia implementación clínica si se cumplen las siguientes condiciones de forma secuencial:

1. **Validación externa (prioridad máxima):** Replicar las métricas (F1 macro ≥ 0.75, recall AA ≥ 0.65) en al menos una cohorte independiente de distinta región geográfica.
2. **Aprobación ética:** Protocolo aprobado por comité de ética institucional que cubra el uso de datos lipidómicos con fines de clasificación diagnóstica asistida.
3. **Integración como apoyo (no reemplazo):** El modelo debe funcionar como sistema de apoyo a la decisión clínica, no como sustituto del juicio médico ni de la colonoscopía confirmatoria.
4. **Monitoreo continuo:** Implementar detección de *data drift* y degradación del rendimiento en producción, con protocolo de reentrenamiento documentado.
5. **Validación del subconjunto Top 15:** Antes de considerar cambios al panel de variables, validar mediante nested CV y cohorte externa si el subconjunto Top 15 conserva Recall AA = 0.833 sin sacrificar desempeño global.

---

## 4. Hallazgos del Negocio y Recomendaciones Accionables

### 4.1 Hallazgos clínicos traducidos desde los resultados técnicos

#### Hallazgo 1: Los perfiles lipidómicos discriminan efectivamente entre estados de progresión del cáncer colorrectal

El modelo Bagging Tree alcanza un AUC macro de 0.945 utilizando únicamente 12 biomarcadores lipidómicos, 2 variables clínicas (edad, FIT) y 2 categóricas (género, FOB). Esto confirma que los perfiles lipidómicos capturan información biológica relevante sobre la progresión CTRL → AA → CRC que no está contenida exclusivamente en las pruebas clínicas estándar.

**Evidencia:** El *feature importance* del Bagging Tree posiciona a **CE(20:5)** (éster de colesterol con 5 insaturaciones) como el biomarcador más discriminativo (importancia = 0.137), seguido de **TG(51:4)** y **PC(O-16:0/16:0)**. Estos lípidos específicos tienen plausibilidad biológica documentada: los ésteres de colesterol están implicados en la remodelación de membranas tumorales y la señalización inflamatoria asociada a neoplasias colorrectales.

#### Hallazgo 2: Los lípidos complementan al FIT, no lo reemplazan

La variable clínica `fit_ug_g` (hemoglobina fecal) aparece en la posición #4 del ranking de importancia del modelo, por detrás de tres biomarcadores lipidómicos. Esto valida la hipótesis de que el panel lipidómico aporta información *complementaria* al FIT, no redundante. Un sistema de screening que combine ambas fuentes sería más potente que cualquiera de las dos por separado.

**Evidencia cuantitativa:** Con solo FIT como variable predictiva, la literatura reporta sensibilidad ~50% para AA. Nuestro modelo multimodal (lípidos + FIT) alcanza recall AA = 75%, una mejora del 50% relativo.

#### Hallazgo 3: El adenoma avanzado (AA) es detectable pero inherentemente ambiguo

El análisis PCA (Avance 1) mostró que las elipses de confianza al 95% de AA y CTRL se solapan significativamente en el espacio lipidómico, mientras que CRC se separa claramente. Esto refleja la realidad biológica: AA es un estado transicional donde solo algunos perfiles metabólicos divergen del control sano. A pesar de esta ambigüedad inherente, el modelo logra F1 AA = 0.78, demostrando que la señal existe pero es sutil.

#### Hallazgo 4: Género y antecedentes de FOB aportan información mínima una vez considerados los lípidos

Las variables `gender_Male` y `fob_YES` tienen importancias cercanas a cero en el modelo final. Esto sugiere que las diferencias de género en perfil lipidómico ya están capturadas por los biomarcadores directos, y que el antecedente de sangre oculta en heces no agrega valor predictivo incremental cuando se dispone del perfil metabólico completo.

#### Hallazgo 5: La mejora en AA puede concentrarse en un subconjunto reducido derivado de X_full

La evaluación complementaria solicitada por la Dra. Grettel mostró que el modelo entrenado con X_full incrementa el Recall AA de 0.750 a 0.833. Sin embargo, este beneficio no requiere conservar las 131 variables originales: el subconjunto **Top 15** de variables más importantes de X_full mantiene el mismo Recall AA (0.833) y el mismo F1 AA (0.833) que X_full completo, con F1 macro prácticamente idéntico (0.839).

**Evidencia:** Las variables más importantes identificadas en X_full incluyen **SM(d18:0/18:0)**, **CE(20:5)**, **TG(51:4)**, **fit_ug_g**, **PC(O-16:0/16:0)** y **CE(20:4)**. Este resultado sugiere que la señal discriminativa para adenomas avanzados se concentra en un conjunto pequeño de biomarcadores y abre la posibilidad de diseñar un panel más compacto para validación clínica futura.


### 4.2 Recomendaciones accionables por *stakeholder*

Las siguientes recomendaciones asignan responsables específicos a cada acción, con entregables y prioridad definidos:

| # | Acción | Responsable principal | Apoyo | Entregable | Prioridad |
|---|---|---|---|---|---|
| **R1** | Validar el modelo con cohorte externa independiente (idealmente n ≥ 200, multicéntrica) | Patrocinador académico + Equipo clínico/laboratorio del centro colaborador | Equipo de ciencia de datos (preparación de datos y ejecución del modelo) | Base de datos validada y reporte comparativo de métricas externas vs. internas | **Alta** |
| **R2** | Regularizar el Bagging Tree (`max_depth=7-10`, `min_samples_leaf=3-5`) y evaluar nuevos hiperparámetros | Equipo de ciencia de datos | Asesor técnico de ML | Modelo ajustado con brecha train-CV < 0.10 y reporte de comparación | **Alta** |
| **R3** | Generar explicabilidad con SHAP (*TreeExplainer*) por paciente | Equipo de *machine learning* | Equipo clínico (validación de plausibilidad biológica) | Reporte interpretativo por predicción individual con visualización SHAP | **Alta** |
| **R4** | Diseñar y validar un panel lipidómico reducido basado en las variables Top 15 derivadas de X_full, priorizando biomarcadores como SM(d18:0/18:0), CE(20:5), TG(51:4), PC(O-16:0/16:0) y CE(20:4), para ensayo MRM/SRM | Equipo biomédico / Laboratorio de espectrometría | Equipo de ML (ranking de importancia y análisis Top-K) | Lista final de biomarcadores candidatos con protocolo analítico estandarizado | **Alta** |
| **R5** | Definir protocolo ético y consentimiento para uso de datos lipidómicos con fines de clasificación diagnóstica | Comité de ética de la institución médica | Patrocinador académico y asesor legal | Protocolo ético aprobado y formato de consentimiento informado | **Alta** |
| **R6** | Implementar prototipo del modelo en plataforma cloud como sistema de *triage* experimental | Equipo técnico / MLOps | *Stakeholders* clínicos (requisitos de interfaz) | API funcional o dashboard de predicción desplegado en Vertex AI | **Media** |
| **R7** | Medir impacto operativo y costo-beneficio: comparar protocolo actual (FIT → colonoscopía) vs. protocolo aumentado (FIT + panel lipidómico → colonoscopía selectiva) | *Stakeholders* clínicos y administrativos del hospital | Equipo de datos (modelado económico) | Informe de viabilidad económica con estimación de colonoscopías evitadas y ahorro proyectado | **Media** |
| **R8** | Validar formalmente el subconjunto Top 15 con validación cruzada anidada y cohorte externa | Equipo de ciencia de datos | Asesora técnica y equipo clínico | Reporte comparativo X_intersect vs. Top 15 con métricas CV, held-out y validación externa | **Alta** |

### 4.3 Detalle de recomendaciones clave

#### R1 — Validación externa multicéntrica

**Fundamento:** El modelo se entrenó y evaluó sobre una única cohorte del Hospital de Ourense (Galicia). Las métricas held-out (n=43) tienen varianza inherente de ±0.02–0.05 F1. Sin validación externa, no es posible confirmar que los biomarcadores seleccionados generalizan a otras poblaciones con diferente dieta, genética y protocolos hospitalarios.

**Acción concreta:** Coordinar con al menos dos centros hospitalarios adicionales (idealmente de diferentes regiones geográficas) para recolectar muestras lipidómicas con el mismo protocolo UHPLC-MS. Evaluar el modelo Bagging Tree serializado sobre estas cohortes sin reentrenamiento.

#### R4 — Panel lipidómico reducido

**Fundamento:** El modelo utiliza solo 12 biomarcadores lipidómicos para alcanzar AUC = 0.945. Un panel de espectrometría de masas dirigida (MRM/SRM) con los 5 lípidos de mayor importancia sería más económico (~€50-100 vs ~€500 por muestra en metabolómica no dirigida) y más rápido, facilitando la escalabilidad del screening.

#### R6 — Prototipo en cloud

**Fundamento:** La implementación como prototipo permite validar la integración técnica (pipeline de preprocesamiento + modelo + interfaz) sin riesgo clínico, identificar cuellos de botella operativos y generar evidencia de usabilidad antes de la fase de validación clínica formal. El detalle de la plataforma cloud seleccionada se presenta en la Sección 5.

### 4.4 Análisis de limitaciones y mitigaciones

| # | Limitación | Impacto | Mitigación propuesta |
|---|---|---|---|
| **L1** | **Tamaño muestral pequeño** (n=211 total, n=43 test) | Métricas held-out con alta varianza (±0.02-0.05 F1); riesgo de sobreajuste a patrones de la cohorte específica | Validación externa multicéntrica; uso de la estimación CV (0.802) como referencia conservadora |
| **L2** | **Cohorte única** (Hospital de Ourense, Galicia) | Posible sesgo poblacional: dieta atlántica, genética ibérica, protocolo hospitalario específico | Replicación con cohortes de diferentes regiones geográficas y composiciones étnicas |
| **L3** | **Diseño transversal** (no longitudinal) | Imposibilidad de modelar la transición CTRL → AA → CRC como proceso dinámico | Estudio prospectivo de seguimiento con muestras seriadas |
| **L4** | **Ambigüedad biológica de AA** | Heterogeneidad metabólica: algunos AA son lipidómicamente indistinguibles de CTRL | Subanálisis por tamaño de adenoma (≥10mm vs <10mm) y grado de displasia |
| **L5** | **Mecanismo de datos faltantes (MAR/MNAR)** | 27 lípidos con >20% de valores faltantes; posible sesgo informativo | Imputación por grupo validada (Frobenius < MICE); exclusión de variables con >40% faltantes |
| **L6** | **Sobreajuste del modelo en entrenamiento** (Train F1 = 1.00) | Brecha train-CV de ~0.20 indica memorización parcial | Aumentar regularización: limitar `max_depth`, incrementar `min_samples_leaf` |
| **L7** | **Análisis Top-K exploratorio sin CV completa** | Top 15 conserva Recall AA = 0.833 en held-out, pero aún no se validó mediante nested CV ni cohorte externa | Repetir experimento Top-K con validación cruzada anidada y validación multicéntrica antes de modificar el modelo final |

---

## 5. Implementación en Nube: Comparativa de Proveedores y Arquitectura para Piloto Controlado

### 5.1 Comparativa de proveedores cloud

Para la implementación del prototipo del modelo Bagging Tree como sistema de *triage* experimental, se evaluaron cuatro proveedores de nube líderes en servicios de *machine learning*:

| Criterio | AWS SageMaker | Azure Machine Learning | Google Vertex AI | IBM watsonx.ai |
|---|---|---|---|---|
| **Facilidad de uso** | Alta, pero requiere mayor configuración técnica inicial | Alta, especialmente si la institución ya opera con ecosistema Microsoft | Alta, muy integrado para flujos de ML modernos con notebooks y pipelines | Media-alta, fuerte en entorno empresarial con gobierno corporativo |
| **Escalabilidad** | Muy alta | Muy alta | Muy alta | Alta |
| **Despliegue de modelos** | Endpoints en tiempo real, inferencia *serverless*, *batch inference* | *Managed online endpoints*, pipelines y MLOps integrados | Endpoints, pipelines, AutoML, Model Registry | Despliegue ML y modelos en watsonx.ai con gobierno integrado |
| **Costos** | Pago por uso; inferencia *serverless* puede escalar a cero en baja demanda, útil para prototipos con tráfico bajo | Sin cargo adicional por el servicio base; cargos por cómputo y servicios consumidos | Cargos por entrenamiento, despliegue en endpoint y predicciones, facilitando estimación por etapa del ciclo ML | Planes por uso y opciones empresariales; algunos planes estándar incluyen cuota mensual |
| **MLOps** | Muy fuerte (SageMaker Pipelines, Model Monitor, Feature Store) | Muy fuerte (Azure DevOps integrado, ML Pipelines, Responsible AI) | Muy fuerte (Vertex Pipelines, Model Monitoring, Experiments) | Fuerte en gobierno empresarial e IA responsable (AI Factsheets) |
| **Explicabilidad / IA responsable** | SageMaker Clarify para sesgo y explicabilidad | Responsible AI Dashboard integrado | Vertex Explainable AI (feature attributions) | AI Factsheets, OpenScale para monitoreo de sesgo y equidad |
| **Cumplimiento normativo (salud)** | HIPAA eligible, BAA disponible | HIPAA, HITRUST, certificaciones de salud amplias | HIPAA eligible con BAA | HIPAA ready, fuerte en industrias reguladas |
| **Adecuación para este proyecto** | Excelente para prototipo técnico escalable | Muy recomendable si la institución hospitalaria usa ecosistema Microsoft | Muy recomendable para notebooks, pipelines experimentales y monitoreo continuo | Útil si se prioriza gobierno empresarial e IA híbrida en entorno corporativo |
| **Riesgo principal** | Curva de aprendizaje en configuración de infraestructura | Costos si se dejan endpoints activos sin control | Costos por endpoints activos y servicios conectados | Puede ser más costoso para prototipos pequeños; menor comunidad open-source |

### 5.2 Selección justificada del proveedor

**Proveedor recomendado: Google Vertex AI**

Se recomienda **Google Vertex AI** como plataforma de implementación inicial del prototipo, basándose en los siguientes criterios ponderados según las necesidades específicas del proyecto:

| Factor de decisión | Peso | Justificación para Vertex AI |
|---|---|---|
| **Flujo experimental con notebooks** | Alto | Vertex AI Workbench permite transición directa desde Jupyter notebooks (entorno actual del proyecto) hacia pipelines productivos sin reescritura de código. El equipo ya trabaja en notebooks; la curva de aprendizaje es mínima. |
| **Pipeline reproducible** | Alto | Vertex Pipelines (basado en Kubeflow) permite codificar el pipeline completo (imputación → Box-Cox → selección de features → Bagging Tree → predicción) como un DAG versionado, alineado con la trazabilidad CRISP-ML(Q) del proyecto. |
| **Monitoreo de *drift*** | Alto | Vertex Model Monitoring detecta automáticamente *data drift* y *prediction drift*, crucial para un modelo biomédico donde cambios en protocolo analítico o población podrían degradar el rendimiento silenciosamente. |
| **Costos para prototipo** | Medio | Vertex AI cobra por etapa del ciclo ML (entrenamiento, endpoint, predicciones), lo que facilita estimar y controlar costos en un prototipo académico con tráfico bajo e intermitente. |
| **Explicabilidad integrada** | Medio | Vertex Explainable AI proporciona *feature attributions* por predicción, complementando el análisis SHAP propuesto en R3. |
| **Escalabilidad futura** | Medio | Si el piloto avanza a validación multicéntrica, Vertex AI escala sin cambio de arquitectura. |

**Alternativas consideradas:**

- **AWS SageMaker** sería igualmente robusto y ofrece inferencia *serverless* (escala a cero sin tráfico), pero requiere mayor configuración de infraestructura. Es una alternativa sólida si el equipo tiene experiencia previa con AWS.
- **Azure Machine Learning** sería la opción preferida si la institución hospitalaria ya opera sobre ecosistema Microsoft (Active Directory, Azure DevOps, Teams), por la integración nativa con herramientas institucionales.
- **IBM watsonx.ai** sería más conveniente en un escenario corporativo con fuerte gobierno de IA e infraestructura híbrida (on-premise + cloud), pero resulta menos práctico para un prototipo académico experimental.

### 5.3 Arquitectura propuesta para piloto controlado en entorno productivo experimental

La siguiente arquitectura define los componentes necesarios para el prototipo piloto en Google Vertex AI:

| Componente | Propuesta | Detalle |
|---|---|---|
| **Entrada de datos** | Archivo clínico/lipidómico anonimizado | CSV o JSON con las 16 variables del modelo final X_intersect (12 lípidos + age + fit_ug_g + gender_Male + fob_YES). Para fase experimental, se recomienda conservar adicionalmente las variables Top 15 derivadas de X_full como panel candidato de comparación. Datos anonimizados sin identificadores de paciente. |
| **Preprocesamiento** | Pipeline automatizado | Imputación por mediana de grupo → transformación Box-Cox (λ precomputados) → StandardScaler (μ, σ precomputados) → selección de las 16 features X_intersect. Todos los parámetros fijados desde el entrenamiento, sin recomputar. |
| **Modelo** | Bagging Tree serializado (`joblib`/`pickle`) | Modelo final X_intersect con control de versión (Git + Model Registry de Vertex AI). En etapa de investigación puede registrarse un modelo candidato Top 15 en modo comparativo/*shadow*, documentando métricas, hiperparámetros y fecha de entrenamiento. |
| **Servicio de predicción** | Endpoint en Vertex AI | Endpoint online para predicciones individuales (latencia < 200ms) y *batch prediction* para análisis de cohortes completas. |
| **Interfaz** | Dashboard clínico o API interna REST | Interfaz web para que el clínico ingrese datos del paciente y reciba: (a) clase predicha (CTRL/AA/CRC), (b) probabilidades por clase, (c) explicación SHAP de los factores principales. |
| **Seguridad** | Protección multinivel | Datos anonimizados (sin PII), control de acceso basado en roles (IAM), cifrado en tránsito (TLS) y en reposo, bitácora de auditoría de cada predicción. |
| **Monitoreo** | Vertex Model Monitoring | Detección automática de *data drift* (cambios en distribución de las 16 features), *prediction drift* (cambios en distribución de probabilidades), alertas por caída en tasa de predicción de AA. |
| **Explicabilidad** | SHAP por predicción | Cada predicción incluye los 5 *features* que más contribuyeron a la clasificación del paciente, permitiendo al clínico evaluar la plausibilidad biológica caso por caso. |
| **Reentrenamiento** | Solo con validación aprobada | El modelo no se reentrena automáticamente. Cada actualización requiere: (a) nueva cohorte documentada, (b) validación cruzada con métricas ≥ modelo anterior, (c) aprobación del equipo clínico. |

#### Diagrama de flujo del pipeline piloto

```
┌─────────────────┐     ┌──────────────────────┐     ┌─────────────────┐
│  Datos clínicos  │────▶│  Preprocesamiento     │────▶│  Bagging Tree   │
│  anonimizados    │     │  (imputación, Box-Cox, │     │  (100 árboles)  │
│  (16 variables)  │     │   scaling, selección)  │     │                 │
└─────────────────┘     └──────────────────────┘     └────────┬────────┘
                                                              │
                                                              ▼
                         ┌──────────────────────┐     ┌─────────────────┐
                         │  Explicabilidad SHAP  │◀────│  Predicción     │
                         │  (top 5 features por  │     │  (CTRL/AA/CRC + │
                         │   paciente)           │     │   probabilidades)│
                         └──────────────────────┘     └────────┬────────┘
                                                              │
                                                              ▼
                         ┌──────────────────────┐     ┌─────────────────┐
                         │  Monitoreo continuo   │◀────│  Dashboard      │
                         │  (drift, rendimiento, │     │  clínico / API  │
                         │   alertas)            │     │                 │
                         └──────────────────────┘     └─────────────────┘
```

> **Nota importante:** La implementación propuesta **no sustituye el juicio clínico ni la colonoscopía confirmatoria**; funciona como sistema de apoyo a la decisión y herramienta de priorización experimental para investigación. El sistema debe considerarse exclusivamente como herramienta de apoyo experimental y no como dispositivo médico validado.

---

## 6. Reflexión sobre el Proceso CRISP-ML(Q) y Lecciones Aprendidas

### 6.1 Reflexión crítica por fase

#### Fase 1 — Comprensión del Negocio y los Datos (Avance 1)

**Decisión clave:** Definir el problema como **multiclase** (CTRL/AA/CRC) en lugar de binario (sano/enfermo).

**Impacto:** Esta decisión fue fundamental para todo el proyecto. Un enfoque binario habría colapsado AA con CRC o con CTRL, perdiendo la oportunidad de detectar la clase de mayor valor preventivo. Aunque complicó todas las fases posteriores (métricas macro, desbalance de clases, ambigüedad lipidómica de AA), la formulación multiclase alinea el modelo con la realidad clínica de la progresión del cáncer colorrectal.

**Reflexión:** El EDA exhaustivo de 5 pasos (calidad, distribuciones, transformaciones, ranking supervisado, estructura multivariada) fue esencial para detectar patrones no obvios, como el mecanismo MNAR de datos faltantes (lípidos con concentraciones bajo el límite de detección más frecuentes en CTRL) y la alta correlación intra-familia (mediana |r| = 0.432). Sin este diagnóstico, la fase de modelado habría sido ciega a estos sesgos.

#### Fase 2 — Preparación de Datos (Avance 2)

**Decisión clave:** Seleccionar **imputación por mediana de grupo** sobre MICE, validada con norma de Frobenius (3.24 vs 13.58).

**Impacto:** MICE, siendo un método más sofisticado, habría destruido la estructura de correlación lipidómica porque sus modelos de regresión no incorporan la variable grupo como condicionante. La validación cuantitativa (norma de Frobenius 4× menor) proporcionó evidencia objetiva para una decisión contraintuitiva: a veces, el método más simple es el más apropiado.

**Decisión clave:** Generar **cuatro matrices candidatas** (X_full, X_corr, X_top_mi, X_top_anova) y su intersección (X_intersect).

**Impacto:** La estrategia de intersección MI ∩ ANOVA garantizó que los 16 *features* seleccionados fueran robustos bajo dos criterios independientes (no lineal y lineal). Esto previno la selección de variables artificialmente discriminativas por un solo método. En Avance 4, la comparación X_intersect vs X_full confirmó empíricamente la superioridad de la selección parsimoniosa (F1 held-out: 0.681 vs 0.640; overfitting 2× menor).

#### Fase 3 — Modelado Baseline (Avance 3)

**Decisión clave:** Usar Regresión Logística como baseline en lugar de un modelo más potente.

**Impacto:** Establecer un baseline interpretable y estable (F1 macro = 0.628) proporcionó un punto de referencia claro contra el cual medir todas las mejoras posteriores. La decisión de usar `class_weight='balanced'` ya en esta fase reveló la mejora de recall AA (0.33 → 0.47), una señal temprana de que el desbalance de clases requería atención explícita en todos los modelos.

**Reflexión:** Seleccionar F1 macro como métrica primaria desde esta fase fue acertado: evitó el sesgo de accuracy (que favorecería las clases mayoritarias CTRL/CRC) y mantuvo la evaluación centrada en AA, la clase de mayor valor clínico.

#### Fase 4 — Modelado y Selección Individual (Avance 4)

**Decisión clave:** Implementar evaluación en **dos niveles** (CV sobre X_dev + held-out sobre X_test).

**Impacto:** Esta separación estricta previno el *data leakage* y proporcionó estimaciones no sesgadas del rendimiento. La comparación de 6 algoritmos con y sin tuning demostró que KNN tenía F1-CV ligeramente superior (0.729 vs 0.707) pero Decision Tree generalizaba mejor y ofrecía interpretabilidad clínica. La decisión de priorizar interpretabilidad sobre performance marginal (+0.02 F1) fue correcta dado el dominio biomédico del proyecto.

**Reflexión:** La validación de la sugerencia de Dra. Grettel (probar X_full con el modelo seleccionado) demostró rigurosidad científica: en lugar de asumir que más features = mejor modelo, se evaluó empíricamente y se documentó que X_full genera 2× más overfitting sin mejorar F1 held-out.

#### Fase 5 — Ensembles y Modelo Final (Avance 5)

**Decisión clave:** Evaluar tanto ensembles **homogéneos** (5 modelos) como **heterogéneos** (2 modelos), seleccionando Bagging Tree.

**Impacto:** La comparación sistemática de 8 ensembles reveló que los homogéneos (basados en árboles) superaron consistentemente a los heterogéneos (Stacking, Voting), probablemente porque la combinación de modelos con paradigmas diferentes (KNN + LogReg + SVM) no captura las interacciones no lineales entre lípidos tan eficazmente como múltiples árboles especializados. Bagging Tree, con su mecanismo de *bootstrap* + subsampling de features, logró la mejor reducción de varianza respecto al árbol individual: F1 +0.174.

**Reflexión:** El análisis de feature importance del Bagging Tree cerró el ciclo interpretativo del proyecto: CE(20:5) como biomarcador principal tiene plausibilidad biológica (metabolismo de EPA en carcinogénesis) y coincide con hallazgos de Albóniga et al. (2025), validando externamente la relevancia de los features seleccionados.

### 6.2 Lecciones aprendidas

#### Lección 1: La validación cuantitativa de decisiones de preprocesamiento es tan importante como la validación del modelo

**Evidencia:** En Avance 2, la decisión entre imputación por mediana de grupo y MICE no se tomó por conveniencia sino por validación formal (norma de Frobenius 3.24 vs 13.58, pares con |Δr| > 0.10: 1.6% vs 15.2%). Si hubiéramos elegido MICE sin validar, la estructura de correlación lipidómica se habría distorsionado silenciosamente, propagando sesgos invisibles a todas las fases de modelado.

**Implicación general:** En proyectos de *machine learning* biomédico, cada decisión de preprocesamiento (imputación, transformación, escalado) debe documentarse con métricas de validación, no solo con argumentos teóricos. El pipeline completo es tan confiable como su eslabón más débil.

#### Lección 2: La parsimonia de features es una ventaja, no una limitación

**Evidencia:** X_intersect (16 features) superó consistentemente a X_full (131 features) en todas las métricas held-out a lo largo de Avances 4 y 5:
- Avance 4 (Decision Tree): F1 held-out 0.681 vs 0.640 (+6.4%)
- Avance 5 (Bagging Tree): F1 held-out 0.855 vs 0.839 (+1.9%)
- Análisis Top-K: Top 15 conserva el mismo Recall AA (0.833) y F1 AA (0.833) que X_full completo con 89% menos variables.
- Gap train-CV: 0.20 vs 0.32 (38% menos overfitting)

**Implicación general:** Con muestras pequeñas (n=211) y alta dimensionalidad (127 lípidos), la selección supervisada de features no solo mejora el rendimiento sino que reduce el riesgo de sobreajuste. La máxima "más datos es mejor" no aplica a features cuando n << p; la selección rigurosa (MI ∩ ANOVA) actúa como una forma de regularización implícita. Además, el análisis Top-K muestra que la señal adicional de X_full para AA puede concentrarse en pocos biomarcadores, lo cual debe validarse formalmente antes de adoptarse.

#### Lección 3: La clase difícil define el techo del modelo, y su mejora requiere estrategias específicas

**Evidencia:** AA fue consistentemente la clase más difícil:
- Baseline LogReg: F1 AA = 0.38 (vs CTRL 0.72, CRC 0.68)
- Decision Tree: F1 AA = ~0.55 (vs CTRL ~0.72, CRC ~0.68)
- Bagging Tree: F1 AA = 0.78 (vs CTRL 0.93, CRC 0.85)

La mejora más grande vino del ensemble (+0.23 en F1 AA), no del cambio de algoritmo individual. Esto indica que la heterogeneidad de AA se beneficia de la agregación de múltiples perspectivas (100 árboles con subconjuntos aleatorios) más que de un algoritmo individual más sofisticado.

**Implicación general:** En problemas multiclase con una clase inherentemente ambigua, la estrategia de modelado debe priorizar mecanismos de reducción de varianza (ensembles, regularización) sobre la complejidad del clasificador base. La clase difícil no se resuelve con más parámetros, sino con más estabilidad.

#### Lección 4: La trazabilidad CRISP-ML(Q) habilita decisiones informadas en fases posteriores

**Evidencia:** En Avance 5, la selección final entre Bagging Tree y Random Forest se fundamentó en datos documentados de Avances anteriores:
- La coherencia con la familia Decision Tree (Avance 4) se pudo verificar porque el árbol individual estaba completamente documentado.
- La comparación X_intersect vs X_full (Avance 4) se reutilizó directamente para justificar la matriz de features en Avance 5.
- El baseline de F1 = 0.628 (Avance 3) proporcionó el umbral mínimo para cualquier modelo posterior.

**Implicación general:** Sin la documentación explícita de calidad gates, matrices de riesgo y métricas por fase, las decisiones en fases avanzadas habrían sido *ad hoc*. CRISP-ML(Q) transformó la documentación de un requisito burocrático a una herramienta operativa de toma de decisiones.

### 6.3 Mejoras concretas para iteraciones futuras

| # | Mejora propuesta | Fase CRISP-ML(Q) | Justificación basada en hallazgos |
|---|---|---|---|
| **M1** | Regularización del Bagging Tree: limitar `max_depth=7-10` y `min_samples_leaf=3-5` | Modelado | Train F1=1.00 indica memorización; regularización explícita reduciría la brecha train-CV |
| **M2** | *Oversampling* sintético (SMOTE) específico para AA | Preparación de datos | AA es la clase minoritaria (27.5%) y más heterogénea; generar muestras sintéticas en la frontera AA/CTRL podría mejorar el recall |
| **M3** | Validación cruzada anidada (nested CV) | Evaluación | Evitar sesgo optimista en la selección de hiperparámetros; proporciona estimación más robusta del error de generalización |
| **M4** | Explicabilidad con SHAP (TreeExplainer) | Evaluación/Despliegue | Feature importance por Gini es global; SHAP proporcionaría explicaciones *por paciente*, crucial para adopción clínica |
| **M5** | Validación externa con cohorte independiente | Evaluación | Confirmar que los biomarcadores seleccionados generalizan fuera de la cohorte de Ourense |
| **M6** | Incorporar variables ómicas adicionales (proteómica, microbioma) | Comprensión de datos | El perfil lipidómico captura una dimensión del fenotipo; otros ómicos podrían mejorar la distinción AA/CTRL |
| **M7** | Optimización de umbral de decisión por clase | Modelado | Calibrar las probabilidades predichas para maximizar recall AA sin sacrificar excesivamente la precisión global |
| **M8** | Validar y refinar el subconjunto Top 15 derivado de X_full | Modelado/Evaluación | Top 15 conserva Recall AA = 0.833 y F1 AA = 0.833 en held-out, pero requiere nested CV y validación externa antes de considerarse alternativa al modelo final |

---

## 7. Conclusión Final

Este proyecto demuestra que los biomarcadores lipidómicos, analizados mediante un ensemble de árboles de decisión (Bagging Tree), permiten clasificar con alta eficacia los tres estados de progresión del cáncer colorrectal (CTRL, AA, CRC) utilizando únicamente 16 variables. Los resultados principales son:

1. **Rendimiento clínicamente relevante:** F1 macro = 0.855 (held-out), AUC = 0.945, con detección de adenomas avanzados (F1 AA = 0.78) muy superior al estándar PLS-DA de referencia (F1 AA = 0). Como análisis complementario, Top 15 derivado de X_full alcanzó Recall AA = 0.833 y F1 AA = 0.833, identificándose como candidato para validación futura.

2. **Todos los criterios de éxito definidos en Fase 0 fueron alcanzados o superados**, con brechas positivas en todas las métricas clave. La estimación conservadora vía validación cruzada (F1 CV = 0.802) también supera ampliamente los umbrales mínimos.

3. **Decisión de implementación madura:** No se recomienda implementación clínica inmediata como herramienta diagnóstica autónoma; sí se recomienda una implementación piloto controlada como sistema de apoyo a la investigación, sujeto a validación externa, revisión ética y validación clínica prospectiva. El sistema debe considerarse exclusivamente como herramienta de apoyo experimental y no como dispositivo médico validado para toma de decisiones clínicas.

4. **CE(20:5) se posiciona como biomarcador candidato principal** para screening lipidómico de cáncer colorrectal, con plausibilidad biológica (metabolismo de EPA/omega-3) y validación computacional como el *feature* más discriminativo del modelo.

5. **Google Vertex AI como plataforma de implementación** ofrece el equilibrio óptimo entre facilidad de uso (notebooks nativos), capacidades MLOps (pipelines, monitoreo de *drift*, Model Registry) y escalabilidad para la transición de prototipo académico a validación multicéntrica.

6. **El proceso CRISP-ML(Q) proporcionó el marco necesario** para tomar decisiones informadas, documentadas y reproducibles en cada fase, desde la validación cuantitativa de la imputación hasta la selección del ensemble final.

7. **Las limitaciones identificadas** (tamaño muestral, cohorte única, ambigüedad biológica de AA y carácter exploratorio del análisis Top-K) son abordables en iteraciones futuras y no invalidan los hallazgos actuales, sino que definen la agenda de investigación para la validación clínica.

8. **El análisis Top-K posterior a la retroalimentación docente** mostró que la mejora en la detección de AA observada con X_full puede preservarse usando solo 15 predictores, lo que abre una ruta concreta para mejorar interpretabilidad y viabilidad clínica sin depender de las 131 variables originales.

El proyecto valida la hipótesis central de que los perfiles lipidómicos **complementan** (no reemplazan) las pruebas de screening estándar como el FIT, abriendo una vía prometedora para la detección temprana de lesiones precancerosas mediante análisis metabolómico combinado con *machine learning* interpretativo.

La implementación propuesta no sustituye el juicio clínico ni la colonoscopía confirmatoria; funciona como sistema de apoyo a la decisión y herramienta de priorización experimental.

---

## 8. Referencias

Albóniga, O. E., Cubiella, J., Bujanda, L., Aspichueta, P., Blanco, M. E., Lanza, B., Alonso, C., & Falcón-Pérez, J. M. (2025). Metabolic signature in combination with fecal immunochemical test as a non-invasive tool for advanced colorectal neoplasia diagnosis. *Cancers*, *17*(14), 2339. https://doi.org/10.3390/cancers17142339

Amazon Web Services. (2025). *Amazon SageMaker: Build, train, and deploy machine learning models*. AWS Documentation. https://docs.aws.amazon.com/sagemaker/

Breiman, L. (1996). Bagging predictors. *Machine Learning*, *24*(2), 123–140. https://doi.org/10.1007/BF00058655

Google Cloud. (2025). *Vertex AI documentation*. Google Cloud. https://cloud.google.com/vertex-ai/docs

IBM. (2025). *IBM watsonx.ai: Enterprise AI and machine learning platform*. IBM Documentation. https://www.ibm.com/products/watsonx-ai

Korolov, M. (2022). Measuring the business impact of AI. *CIO*. https://www.cio.com/article/measuring-the-business-impact-of-ai/

Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems*, *30*, 4765–4774.

Microsoft. (2025). *Azure Machine Learning documentation*. Microsoft Learn. https://learn.microsoft.com/en-us/azure/machine-learning/

Miller, G. (2022). Stakeholder roles in artificial intelligence projects. *Project Leadership and Society*, *3*, 100068. https://doi.org/10.1016/j.plas.2022.100068

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*, 2825–2830.

Studer, S., Bui, T. B., Drescher, C., Hanuschkin, A., Winkler, L., Peters, S., & Müller, K.-R. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction*, *3*(2), 392–413. https://doi.org/10.3390/make3020020

Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

---

*Avance 6 completado — Equipo 22, Junio 2026*